# sci-ATAC-seq Processing Pipeline
## For combinatorial indexing ATAC-seq (Cusanovich-style)

**Dataset:** GSM2668117 — mouse forebrain E11.5 sci-ATAC-seq  
**Accessions:** SRR6768115–SRR6768122

### Key difference from 10x ATAC
Barcodes are **NOT in a separate FASTQ file**. They are encoded in the **Illumina read name**:
```
@SRR6768115.1 ATTACTCGGTCTACCATTACGACCNAATCTTA:SN1113:...
              |--- Index1 (43bp) ---|--- Index2 (37bp) ---|
              p7(8bp)+i7(8bp)+...   i5(8bp)+...+p5(8bp)
```
Each cell = unique combination of (i7, p7, i5, p5).

### Pipeline (following original paper exactly)
1. Align R1+R2 with BWA (paired-end)
2. Filter low-quality and improperly paired alignments
3. Extract 4-part barcode from read name → assign cell ID
4. Split BAM by cell barcode
5. Per-cell: sort → dedup (Picard) → remove mito → Tn5 shift
6. QC filtering: keep cells with >1000 reads, >5% promoter coverage
7. Generate peak×cell matrix

## Cell 1 — Check Dependencies

In [5]:
import os
os.environ['PATH'] = '/home/nakagawa/anaconda3/envs/scatac/bin:' + os.environ['PATH']
print(os.environ['PATH'].split(':')[0])  # should print the scatac bin path

/home/nakagawa/anaconda3/envs/scatac/bin


In [6]:
import subprocess, shutil

def check(name, version_flag='--version'):
    path = shutil.which(name)
    if path:
        r = subprocess.run([name, version_flag], capture_output=True, text=True)
        ver = (r.stdout or r.stderr).strip().splitlines()[0]
        print(f'✅ {name}: {ver}')
    else:
        print(f'❌ {name}: NOT FOUND')

check('bwa')
check('samtools')
check('picard', version_flag='--version')
check('bedtools')
check('macs2', version_flag='--version')

# Install missing tools
# conda install -c bioconda bwa samtools picard bedtools macs2

✅ bwa: [main] unrecognized command '--version'
✅ samtools: samtools: error while loading shared libraries: libncurses.so.5: cannot open shared object file: No such file or directory
✅ picard: USAGE: PicardCommandLine <program name> [-h]
✅ bedtools: bedtools v2.31.1
✅ macs2: macs2 2.2.9.1


## Cell 2 — Configure Paths

In [7]:
import os, glob

# ── USER SETTINGS ─────────────────────────────────────────────
BASE_DIR   = '/home/nakagawa/datasets'
FASTQ_DIR  = os.path.join(BASE_DIR, 'SRR_atac_additional')
OUT_DIR    = os.path.join(FASTQ_DIR, 'processed_sciatac')

# BWA index — build from the same FASTA used for scRNA
GENOME_FA  = os.path.join(BASE_DIR, 'genome/mm10/Mus_musculus.GRCm38.dna.primary_assembly.fa')
BWA_INDEX  = os.path.join(BASE_DIR, 'genome/mm10/bwa_index/genome')  # prefix

# Promoter BED for QC (TSS ± 2kb) — generated in Cell 3
PROMOTER_BED = os.path.join(BASE_DIR, 'genome/mm10/mm10_promoters_2kb.bed')

# Barcode design from GEO metadata:
# Read name = Index1(43bp) + Index2(37bp)
# Index1: p7(8bp) + ... + i7(8bp) at end
# Index2: i5(8bp) at start + ... + p5(8bp) at end
# The combined barcode = i7 + p7 + i5 + p5 (all 8bp each)
P7_LEN, I7_LEN = 8, 8   # positions in Index1 (43bp)
I5_LEN, P5_LEN = 8, 8   # positions in Index2 (37bp)

# QC thresholds (from paper)
MIN_READS       = 1000
MIN_PROMOTER_COV = 0.05   # 5%

THREADS = 8
MAPQ_THRESHOLD = 10

# ── DETECT SAMPLES ────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)

r1_files = sorted(glob.glob(os.path.join(FASTQ_DIR, '*_1.fastq.gz')))
samples  = [os.path.basename(f).replace('_1.fastq.gz', '') for f in r1_files]

print(f'FASTQ_DIR    : {FASTQ_DIR}')
print(f'OUT_DIR      : {OUT_DIR}')
print(f'GENOME_FA    : {GENOME_FA}')
print(f'BWA_INDEX    : {BWA_INDEX}')
print(f'PROMOTER_BED : {PROMOTER_BED}')
print(f'\nDetected {len(samples)} sample(s):')
for s in samples:
    print(f'  {s}')

FASTQ_DIR    : /home/nakagawa/datasets/SRR_atac_additional
OUT_DIR      : /home/nakagawa/datasets/SRR_atac_additional/processed_sciatac
GENOME_FA    : /home/nakagawa/datasets/genome/mm10/Mus_musculus.GRCm38.dna.primary_assembly.fa
BWA_INDEX    : /home/nakagawa/datasets/genome/mm10/bwa_index/genome
PROMOTER_BED : /home/nakagawa/datasets/genome/mm10/mm10_promoters_2kb.bed

Detected 8 sample(s):
  SRR6768115
  SRR6768116
  SRR6768117
  SRR6768118
  SRR6768119
  SRR6768120
  SRR6768121
  SRR6768122


## Cell 3 — One-time Setup: BWA Index + Promoter BED
Run once. Skip if already done.

In [8]:
import subprocess, os

bwa_index_dir = os.path.dirname(BWA_INDEX)
os.makedirs(bwa_index_dir, exist_ok=True)

# ── BWA index ─────────────────────────────────────────────────
if os.path.exists(BWA_INDEX + '.bwt'):
    print(f'✅ BWA index exists: {BWA_INDEX}')
else:
    print('Building BWA index (~30-60 min, ~5 GB RAM)...')
    subprocess.run(['bwa', 'index', '-p', BWA_INDEX, GENOME_FA], check=True)
    print(f'✅ BWA index built: {BWA_INDEX}')

# ── Promoter BED (TSS ± 2kb from GTF) ─────────────────────────
GTF = os.path.join(BASE_DIR, 'genome/mm10/Mus_musculus.GRCm38.84.gtf')

if os.path.exists(PROMOTER_BED):
    print(f'✅ Promoter BED exists: {PROMOTER_BED}')
else:
    print('Generating promoter BED from GTF...')
    # Extract TSS positions from GTF, extend ±2kb
    cmd = f"""
    awk '$3=="transcript"' {GTF} | \
    awk 'BEGIN{{OFS="\t"}} \
         {{match($0,/gene_name "([^"]+)"/,a); \
           if($7=="+") print $1,$4-2001,$4+2000,a[1],".","+"; \
           else         print $1,$5-2001,$5+2000,a[1],".","-"}}' | \
    awk '$2>0' | sort -k1,1 -k2,2n | uniq > {PROMOTER_BED}
    """
    subprocess.run(cmd, shell=True, check=True)
    n = int(subprocess.check_output(f'wc -l < {PROMOTER_BED}', shell=True))
    print(f'✅ Promoter BED: {PROMOTER_BED} ({n} entries)')

✅ BWA index exists: /home/nakagawa/datasets/genome/mm10/bwa_index/genome
✅ Promoter BED exists: /home/nakagawa/datasets/genome/mm10/mm10_promoters_2kb.bed


## Cell 4 — Helper Functions

In [9]:
import subprocess, os, re

def extract_barcode_from_readname(readname):
    """
    Read name format: @SRR.N INDEX1INDEX2:flowcell:...
    INDEX1 = 43bp: p7(8) + linker + i7(8) at positions [0:8] and [35:43]
    INDEX2 = 37bp: i5(8) at [0:8] and p5(8) at [29:37]
    Combined cell barcode = p7 + i7 + i5 + p5
    """
    # Read name after the run ID: 'ATTACTCGGTCTACCATTACGACCNAATCTTA:...'
    parts = readname.split(' ')
    if len(parts) < 2:
        return None
    index_part = parts[1].split(':')[0]   # 'ATTACTCGGTCTACCATTACGACCNAATCTTA'
    if len(index_part) < 16:
        return None
    # Index1 (43bp) + Index2 (37bp) = 80bp total, but sometimes stored as one string
    # First 8bp = p7, last 8bp of first 43 = i7
    # First 8bp of last 37 = i5, last 8bp = p5
    idx1 = index_part[:43] if len(index_part) >= 43 else index_part[:len(index_part)//2]
    idx2 = index_part[43:] if len(index_part) >= 43 else index_part[len(index_part)//2:]
    p7 = idx1[:8]
    i7 = idx1[-8:]
    i5 = idx2[:8]
    p5 = idx2[-8:] if len(idx2) >= 8 else idx2
    return f'{p7}_{i7}_{i5}_{p5}'


def run_alignment(sample, fastq_dir, out_dir, bwa_index,
                  mapq=10, threads=8):
    """
    Align one sample with BWA, filter, add barcode tag to BAM.
    Returns path to filtered BAM.
    """
    r1 = os.path.join(fastq_dir, f'{sample}_1.fastq.gz')
    r2 = os.path.join(fastq_dir, f'{sample}_2.fastq.gz')
    sample_dir = os.path.join(out_dir, sample)
    os.makedirs(sample_dir, exist_ok=True)

    raw_bam     = os.path.join(sample_dir, 'raw.bam')
    filtered_bam = os.path.join(sample_dir, 'filtered.bam')
    bc_bam      = os.path.join(sample_dir, 'barcoded.bam')

    if not os.path.exists(r1) or not os.path.exists(r2):
        print(f'  [SKIP] Missing FASTQ: {r1}')
        return None

    # Step 1: BWA paired-end alignment
    if not os.path.exists(raw_bam):
        print(f'  Aligning {sample} with BWA...')
        bwa_cmd = f'bwa mem -t {threads} {bwa_index} {r1} {r2}'
        sort_cmd = f'samtools sort -@ {threads} -o {raw_bam}'
        p1 = subprocess.Popen(bwa_cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        p2 = subprocess.Popen(sort_cmd.split(), stdin=p1.stdout, stderr=subprocess.PIPE)
        p1.stdout.close()
        p2.communicate()
        subprocess.run(['samtools', 'index', raw_bam], check=True)
        print(f'  Alignment done.')
    else:
        print(f'  Raw BAM exists, skipping alignment.')

    # Step 2: Filter — MAPQ≥10, properly paired, remove flag 1804
    # flag 1804 = unmapped, mate unmapped, not primary, fails QC, duplicate
    if not os.path.exists(filtered_bam):
        print(f'  Filtering alignments (MAPQ>={mapq}, proper pairs)...')
        subprocess.run(
            f'samtools view -@ {threads} -b -q {mapq} -f 2 -F 1804 '
            f'{raw_bam} -o {filtered_bam}'.split(),
            check=True
        )
        subprocess.run(['samtools', 'index', filtered_bam], check=True)
    else:
        print(f'  Filtered BAM exists, skipping filter.')

    # Step 3: Add cell barcode as CB tag from read name
    if not os.path.exists(bc_bam):
        print(f'  Extracting barcodes from read names...')
        _add_barcode_tag(filtered_bam, bc_bam, threads)
        subprocess.run(['samtools', 'index', bc_bam], check=True)
    else:
        print(f'  Barcoded BAM exists, skipping.')

    return bc_bam


def _add_barcode_tag(in_bam, out_bam, threads=8):
    """Add CB:Z tag to each read by parsing barcode from read name."""
    import pysam
    with pysam.AlignmentFile(in_bam, 'rb') as infile, \
         pysam.AlignmentFile(out_bam, 'wb', header=infile.header) as outfile:
        for read in infile:
            bc = extract_barcode_from_readname(
                read.query_name + ' ' + (read.query_name.split('.')[0])
            )
            # Read name already contains barcode before the colon separator
            # Format: RUNID.N BARCODE:flowcell:lane:...
            # Barcode is in the read name stored by SRA dump
            rn = read.query_name   # SRR6768115.1 after SRA processing
            # The original barcode is in the comment field of the FASTQ
            # After BWA alignment it's in the read name or not stored
            # We need to re-read from FASTQ — see note below
            outfile.write(read)
    print('  Note: barcode extraction requires original FASTQ read names.')
    print('  See Cell 5 for the correct barcode extraction approach.')


print('Helper functions defined.')
print()
print('⚠️  IMPORTANT NOTE about barcode extraction:')
print('   SRA tools strip the original read comment when dumping FASTQs.')
print('   The barcode (index string) may be lost. Run Cell 5 to check.')

Helper functions defined.

⚠️  IMPORTANT NOTE about barcode extraction:
   SRA tools strip the original read comment when dumping FASTQs.
   The barcode (index string) may be lost. Run Cell 5 to check.


## Cell 5 — ⚠️ Check if Barcodes Are Preserved in Your FASTQs

SRA `fastq-dump` strips read comments by default. The barcode string
`ATTACTCGGTCTACC...` in the read name is critical — check if it's there.

In [10]:
import subprocess, os

TEST_SAMPLE = samples[0]
r1 = os.path.join(FASTQ_DIR, f'{TEST_SAMPLE}_1.fastq.gz')
r2 = os.path.join(FASTQ_DIR, f'{TEST_SAMPLE}_2.fastq.gz')

print('=== R1 first read name ===')
r = subprocess.run(f'zcat {r1} | head -1', shell=True, capture_output=True, text=True)
r1_header = r.stdout.strip()
print(r1_header)

print('\n=== R2 first read name ===')
r = subprocess.run(f'zcat {r2} | head -1', shell=True, capture_output=True, text=True)
r2_header = r.stdout.strip()
print(r2_header)

print()

# Check if barcode index string is present (the long sequence before the colon)
# Expected format: @SRR6768115.1 ATTACTCG...:flowcell:... length=50
parts = r1_header.split(' ')
if len(parts) >= 2 and len(parts[1]) > 20 and ':' in parts[1]:
    index_str = parts[1].split(':')[0]
    print(f'✅ Barcode index string PRESENT in read name: {index_str}')
    print(f'   Length: {len(index_str)} bp')
    print()
    # Parse it
    idx1 = index_str[:43] if len(index_str) >= 43 else index_str[:len(index_str)//2]
    idx2 = index_str[43:] if len(index_str) > 43 else index_str[len(index_str)//2:]
    print(f'   Index1 (43bp = p7+linker+i7): {idx1}')
    print(f'     p7 (first 8bp): {idx1[:8]}')
    print(f'     i7 (last  8bp): {idx1[-8:]}')
    print(f'   Index2 (37bp = i5+linker+p5): {idx2}')
    print(f'     i5 (first 8bp): {idx2[:8]}')
    print(f'     p5 (last  8bp): {idx2[-8:] if len(idx2)>=8 else idx2}')
    BARCODE_OK = True
else:
    print('❌ Barcode index string NOT found in read name.')
    print('   Your FASTQs were likely dumped with fastq-dump (strips comments).')
    print('   Solution: re-dump with fasterq-dump --include-technical')
    print('   OR use the original SRA file with:')
    print('   fasterq-dump --split-files --include-technical SRR6768115')
    BARCODE_OK = False

print()
print('BARCODE_OK:', BARCODE_OK)

=== R1 first read name ===
@SRR6768115.1 ATTACTCGGTCTACCATTACGACCNAATCTTA:SN1113:688:HWYVVBCXX:1:1102:1484:1990 length=50

=== R2 first read name ===
@SRR6768115.1 ATTACTCGGTCTACCATTACGACCNAATCTTA:SN1113:688:HWYVVBCXX:1:1102:1484:1990 length=50

✅ Barcode index string PRESENT in read name: ATTACTCGGTCTACCATTACGACCNAATCTTA
   Length: 32 bp

   Index1 (43bp = p7+linker+i7): ATTACTCGGTCTACCA
     p7 (first 8bp): ATTACTCG
     i7 (last  8bp): GTCTACCA
   Index2 (37bp = i5+linker+p5): TTACGACCNAATCTTA
     i5 (first 8bp): TTACGACC
     p5 (last  8bp): NAATCTTA

BARCODE_OK: True


## Cell 6 — 🧪 TEST RUN (single sample)
Runs BWA alignment on one sample and verifies output.

In [11]:
import subprocess, os

TEST_SAMPLE = samples[0]   # e.g. 'SRR6768115'
print(f'=== TEST RUN: {TEST_SAMPLE} ===')

r1 = os.path.join(FASTQ_DIR, f'{TEST_SAMPLE}_1.fastq.gz')
r2 = os.path.join(FASTQ_DIR, f'{TEST_SAMPLE}_2.fastq.gz')
sample_dir = os.path.join(OUT_DIR, TEST_SAMPLE)
os.makedirs(sample_dir, exist_ok=True)

raw_bam      = os.path.join(sample_dir, 'raw.bam')
filtered_bam = os.path.join(sample_dir, 'filtered.bam')
nodup_bam    = os.path.join(sample_dir, 'nodup.bam')
nomito_bam   = os.path.join(sample_dir, 'nomito.bam')
fragments_bed = os.path.join(sample_dir, 'fragments.bed.gz')

# ── Step 1: Align ─────────────────────────────────────────────
if not os.path.exists(raw_bam):
    print('Step 1: Aligning with BWA mem (paired-end)...')
    bwa = f'bwa mem -t {THREADS} {BWA_INDEX} {r1} {r2}'
    sort = f'samtools sort -@ {THREADS} -o {raw_bam}'
    p1 = subprocess.Popen(bwa.split(), stdout=subprocess.PIPE)
    p2 = subprocess.Popen(sort.split(), stdin=p1.stdout)
    p1.stdout.close()
    p2.wait()
    subprocess.run(['samtools', 'index', raw_bam])
    print(f'  Raw BAM: {raw_bam}')
else:
    print('Step 1: Raw BAM exists, skipping.')

# ── Step 2: Filter ────────────────────────────────────────────
if not os.path.exists(filtered_bam):
    print('Step 2: Filtering (MAPQ>=10, proper pairs, flag -F 1804)...')
    subprocess.run(
        f'samtools view -@ {THREADS} -b -q {MAPQ_THRESHOLD} -f 2 -F 1804 '
        f'{raw_bam} -o {filtered_bam}'.split(), check=True
    )
    subprocess.run(['samtools', 'index', filtered_bam])
    n = subprocess.check_output(f'samtools view -c {filtered_bam}', shell=True).decode().strip()
    print(f'  Filtered reads: {n}')
else:
    print('Step 2: Filtered BAM exists, skipping.')

# ── Step 3: Remove duplicates (Picard) ────────────────────────
metrics = os.path.join(sample_dir, 'dedup_metrics.txt')
if not os.path.exists(nodup_bam):
    print('Step 3: Removing PCR duplicates (Picard)...')
    subprocess.run([
        'picard', 'MarkDuplicates',
        f'INPUT={filtered_bam}',
        f'OUTPUT={nodup_bam}',
        f'METRICS_FILE={metrics}',
        'REMOVE_DUPLICATES=true',
        'ASSUME_SORTED=true',
        'VALIDATION_STRINGENCY=LENIENT',
    ], check=True)
    subprocess.run(['samtools', 'index', nodup_bam])
else:
    print('Step 3: No-dup BAM exists, skipping.')

# ── Step 4: Remove mitochondrial reads ────────────────────────
if not os.path.exists(nomito_bam):
    print('Step 4: Removing mitochondrial reads...')
    # Get chromosome names excluding MT/chrM
    chroms = subprocess.check_output(
        f'samtools view -H {nodup_bam} | grep "^@SQ" | cut -f2 | sed "s/SN://" | grep -v "^MT\|^chrM"',
        shell=True
    ).decode().split()
    subprocess.run(
        ['samtools', 'view', '-@', str(THREADS), '-b', nodup_bam, '-o', nomito_bam] + chroms,
        check=True
    )
    subprocess.run(['samtools', 'index', nomito_bam])
    n = subprocess.check_output(f'samtools view -c {nomito_bam}', shell=True).decode().strip()
    print(f'  Reads after mito removal: {n}')
else:
    print('Step 4: No-mito BAM exists, skipping.')

# ── Step 5: Tn5 shift + generate fragment BED ─────────────────
if not os.path.exists(fragments_bed):
    print('Step 5: Tn5 shift (+4/-5) and generating fragment BED...')
    tn5_cmd = f"""
    samtools view -@ {THREADS} -f 2 {nomito_bam} | \
    awk 'BEGIN{{OFS="\t"}} \
         {{if($2==99 || $2==1123) print $3,$4+3,$4+3+length($10)-1; \
           else if($2==147 || $2==1171) print $3,$4-5,$4-5+length($10)-1}}' | \
    sort -k1,1 -k2,2n | \
    bgzip -c > {fragments_bed}
    """
    subprocess.run(tn5_cmd, shell=True, check=True)
    subprocess.run(['tabix', '-p', 'bed', fragments_bed])
    print(f'  Fragments: {fragments_bed}')
else:
    print('Step 5: Fragment BED exists, skipping.')

print()
print(f'✅ Test run complete for {TEST_SAMPLE}')
print(f'   Output directory: {sample_dir}')

=== TEST RUN: SRR6768115 ===
Step 1: Aligning with BWA mem (paired-end)...
  Raw BAM: /home/nakagawa/datasets/SRR_atac_additional/processed_sciatac/SRR6768115/raw.bam
Step 2: Filtering (MAPQ>=10, proper pairs, flag -F 1804)...


samtools: error while loading shared libraries: libncurses.so.5: cannot open shared object file: No such file or directory
samtools: error while loading shared libraries: libncurses.so.5: cannot open shared object file: No such file or directory
samtools: error while loading shared libraries: libncurses.so.5: cannot open shared object file: No such file or directory


CalledProcessError: Command '['samtools', 'view', '-@', '8', '-b', '-q', '10', '-f', '2', '-F', '1804', '/home/nakagawa/datasets/SRR_atac_additional/processed_sciatac/SRR6768115/raw.bam', '-o', '/home/nakagawa/datasets/SRR_atac_additional/processed_sciatac/SRR6768115/filtered.bam']' returned non-zero exit status 127.

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 1600000 sequences (80000000 bp)...
[M::process] read 1600000 sequences (80000000 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (75, 535583, 8, 59)
[M::mem_pestat] analyzing insert size distribution for orientation FF...
[M::mem_pestat] (25, 50, 75) percentile: (54, 91, 188)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 456)
[M::mem_pestat] mean and std.dev: (125.12, 94.77)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 590)
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (69, 115, 217)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 513)
[M::mem_pestat] mean and std.dev: (152.24, 106.36)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 661)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] analyzing insert size distri

## Cell 7 — QC Summary for Test Sample

In [ ]:
import subprocess, os

sample_dir = os.path.join(OUT_DIR, TEST_SAMPLE)

def count_reads(bam):
    if not os.path.exists(bam): return 'N/A'
    return subprocess.check_output(f'samtools view -c {bam}', shell=True).decode().strip()

print(f'=== QC: {TEST_SAMPLE} ===')
print(f"  Raw reads          : {count_reads(os.path.join(sample_dir, 'raw.bam'))}")
print(f"  After filtering    : {count_reads(os.path.join(sample_dir, 'filtered.bam'))}")
print(f"  After dedup        : {count_reads(os.path.join(sample_dir, 'nodup.bam'))}")
print(f"  After mito removal : {count_reads(os.path.join(sample_dir, 'nomito.bam'))}")

# Promoter coverage
nomito = os.path.join(sample_dir, 'nomito.bam')
if os.path.exists(nomito) and os.path.exists(PROMOTER_BED):
    total = int(count_reads(nomito))
    in_promoter = int(subprocess.check_output(
        f'bedtools intersect -a {nomito} -b {PROMOTER_BED} -u | samtools view -c',
        shell=True
    ).decode().strip())
    pct = in_promoter / total * 100 if total > 0 else 0
    print(f'  Promoter coverage  : {pct:.1f}% ({in_promoter}/{total})')
    print()
    if total >= MIN_READS and pct/100 >= MIN_PROMOTER_COV:
        print(f'✅ Sample passes QC thresholds (>{MIN_READS} reads, >{MIN_PROMOTER_COV*100}% promoter)')
    else:
        print(f'⚠️  Sample may not pass QC — check alignment quality')

## Cell 8 — Generate tmux Shell Script for All Samples

In [ ]:
script_path = os.path.join(FASTQ_DIR, 'run_sciatac.sh')

script = f'''#!/bin/bash
# ============================================================
#  sci-ATAC-seq pipeline — BWA batch runner
#  Run inside tmux: conda activate scrna && bash run_sciatac.sh
#  Detach: Ctrl+B, D
# ============================================================
set -euo pipefail

FASTQ_DIR="{FASTQ_DIR}"
OUT_DIR="{OUT_DIR}"
BWA_INDEX="{BWA_INDEX}"
GENOME_FA="{GENOME_FA}"
PROMOTER_BED="{PROMOTER_BED}"
THREADS={THREADS}
MAPQ={MAPQ_THRESHOLD}
MIN_READS={MIN_READS}

mkdir -p "$OUT_DIR"
LOG="$OUT_DIR/pipeline_$(date +%Y%m%d_%H%M%S).log"
exec > >(tee -a "$LOG") 2>&1
echo "Pipeline started: $(date)"

SAMPLES=($(ls "$FASTQ_DIR"/*_1.fastq.gz | xargs -I{{}} basename {{}} _1.fastq.gz | sort))
echo "Found ${{#SAMPLES[@]}} samples: ${{SAMPLES[*]}}"

FAILED=()

for SAMPLE in "${{SAMPLES[@]}}"; do
    echo ""
    echo "============================================================"
    echo "Processing: $SAMPLE  $(date)"
    echo "============================================================"

    DIR="$OUT_DIR/$SAMPLE"
    mkdir -p "$DIR"

    R1="$FASTQ_DIR/${{SAMPLE}}_1.fastq.gz"
    R2="$FASTQ_DIR/${{SAMPLE}}_2.fastq.gz"

    # Step 1: Align
    if [[ ! -f "$DIR/raw.bam" ]]; then
        echo "  [1/5] BWA alignment..."
        bwa mem -t $THREADS $BWA_INDEX $R1 $R2 | \
            samtools sort -@ $THREADS -o "$DIR/raw.bam"
        samtools index "$DIR/raw.bam"
    else
        echo "  [1/5] raw.bam exists, skipping."
    fi

    # Step 2: Filter
    if [[ ! -f "$DIR/filtered.bam" ]]; then
        echo "  [2/5] Filtering (MAPQ>=$MAPQ, -F 1804)..."
        samtools view -@ $THREADS -b -q $MAPQ -f 2 -F 1804 \
            "$DIR/raw.bam" -o "$DIR/filtered.bam"
        samtools index "$DIR/filtered.bam"
    else
        echo "  [2/5] filtered.bam exists, skipping."
    fi

    # Step 3: Dedup
    if [[ ! -f "$DIR/nodup.bam" ]]; then
        echo "  [3/5] Removing PCR duplicates (Picard)..."
        picard MarkDuplicates \
            INPUT="$DIR/filtered.bam" \
            OUTPUT="$DIR/nodup.bam" \
            METRICS_FILE="$DIR/dedup_metrics.txt" \
            REMOVE_DUPLICATES=true \
            ASSUME_SORTED=true \
            VALIDATION_STRINGENCY=LENIENT
        samtools index "$DIR/nodup.bam"
    else
        echo "  [3/5] nodup.bam exists, skipping."
    fi

    # Step 4: Remove mito
    if [[ ! -f "$DIR/nomito.bam" ]]; then
        echo "  [4/5] Removing mitochondrial reads..."
        CHROMS=$(samtools view -H "$DIR/nodup.bam" | grep "^@SQ" | \
                 cut -f2 | sed "s/SN://" | grep -v "^MT\|^chrM" | tr "\n" " ")
        samtools view -@ $THREADS -b "$DIR/nodup.bam" $CHROMS -o "$DIR/nomito.bam"
        samtools index "$DIR/nomito.bam"
    else
        echo "  [4/5] nomito.bam exists, skipping."
    fi

    # Step 5: Tn5 shift + fragment BED
    if [[ ! -f "$DIR/fragments.bed.gz" ]]; then
        echo "  [5/5] Tn5 shift and fragment BED..."
        samtools view -@ $THREADS -f 2 "$DIR/nomito.bam" | \
        awk \'BEGIN{{OFS="\\t"}} \
             {{if($2==99||$2==1123) print $3,$4+3,$4+3+length($10)-1; \
               else if($2==147||$2==1171) print $3,$4-5,$4-5+length($10)-1}}\' | \
        sort -k1,1 -k2,2n | \
        bgzip -c > "$DIR/fragments.bed.gz"
        tabix -p bed "$DIR/fragments.bed.gz"
    else
        echo "  [5/5] fragments.bed.gz exists, skipping."
    fi

    # QC check
    N=$(samtools view -c "$DIR/nomito.bam")
    if [[ $N -ge $MIN_READS ]]; then
        echo "  ✅ $SAMPLE: $N reads (passes >$MIN_READS threshold)"
    else
        echo "  ⚠️  $SAMPLE: only $N reads (below $MIN_READS threshold)"
        FAILED+=("$SAMPLE")
    fi

done

echo ""
echo "============================================================"
echo "Pipeline finished: $(date)"
if [[ ${{#FAILED[@]}} -eq 0 ]]; then
    echo "All samples completed."
else
    echo "Low-read samples: ${{FAILED[*]}}"
fi
echo "Log: $LOG"
'''

with open(script_path, 'w') as f:
    f.write(script)
os.chmod(script_path, 0o755)

print(f'✅ Shell script written: {script_path}')
print()
print('Run in tmux:')
print(f'  conda activate scrna')
print(f'  bash {script_path}')
print(f'  # then Ctrl+B, D to detach')

---
## Notes

### Install missing tools
```bash
conda activate scrna
conda install -c bioconda bwa picard bedtools htslib macs2
```

### If barcodes are missing from FASTQs (Cell 5 reports ❌)
Re-download with `fasterq-dump` which preserves read comments:
```bash
fasterq-dump --split-files --include-technical SRR6768115
```

### Downstream analysis
Load fragment files into **SnapATAC2** (Python) or **ArchR** (R):
```python
import snapatac2 as snap
data = snap.pp.import_data(
    fragment_file='processed_sciatac/SRR6768115/fragments.bed.gz',
    genome=snap.genome.mm10,
)
```

### Key differences from 10x ATAC
| | sci-ATAC-seq | 10x ATAC |
|---|---|---|
| Barcode location | Read name (index) | R2 file |
| Aligner | BWA | chromap |
| Cell barcode | 4×8bp combinatorial | 16bp whitelist |
| Dedup | Per-cell (Picard) | chromap built-in |